<a href="https://colab.research.google.com/github/amitdey7/MS107/blob/master/Agents_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
!pip install  --upgrade transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.8 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.2.0
    Uninstalling hf-xet-1.2.0:
      Successfully uninstalled hf-xet-1.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57

In [ ]:
!uv pip install openmeteo-requests litellm
!uv pip install requests-cache retry-requests numpy pandas


Using Python 3.12.12 environment at: /usr
Resolved 62 packages in 650ms
Prepared 18 packages in 688ms
Uninstalled 9 packages in 194ms
Installed 18 packages in 86ms
 - aiohttp==3.13.3
 + aiohttp==3.13.4
 - click==8.3.1
 + click==8.1.8
 + fastuuid==0.14.0
 - flatbuffers==25.12.19
 + flatbuffers==25.9.23
 - importlib-metadata==8.7.1
 + importlib-metadata==8.5.0
 + jh2==5.0.11
 - jsonschema==4.26.0
 + jsonschema==4.23.0
 + litellm==1.83.14
 + niquests==3.18.7
 - openai==2.15.0
 + openai==2.24.0
 + openmeteo-requests==1.7.5
 + openmeteo-sdk==1.26.0
 - pydantic==2.12.3
 + pydantic==2.12.5
 - pydantic-core==2.41.4
 + pydantic-core==2.41.5
 - python-dotenv==1.2.1
 + python-dotenv==1.2.2
 + qh3==1.8.0
 + urllib3-future==2.19.913
 + wassima==2.0.6
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 153ms
Prepared 4 packages in 22ms
Installed 4 packages in 10ms
 + cattrs==26.1.0
 + requests-cache==1.3.1
 + retry-requests==2.0.0
 + url-normalize==3.0.0


## Check weather with a LLM

In [ ]:
from litellm import completion
import json
import os
from google.colab import userdata

# Set your Gemini API key from Colab's user data
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

query = "Tell me what's the weather like in Bangalore today?"
messages = [{"role": "user", "content": query}]

response = completion(
    model="gemini/gemini-flash-latest",
    messages=messages,
    reasoning_effort="medium"
)


print(response.choices[0].message.content)

As of today, **Monday, May 20, 2024**, the weather in Bangalore (Bengaluru) is warm with a mix of sun and clouds.

Here are the details:

*   **Temperature:** High of **31°C (88°F)** and a low of **21°C (70°F)**.
*   **Conditions:** The day is partly cloudy. While it is mostly dry during the day, there is a **40% chance of isolated thunderstorms or light rain** in the late evening or night, which is typical for Bangalore this time of year.
*   **Humidity:** Around 55%.
*   **Wind:** Light breezes from the West at about 15 km/h.

**Summary:** It’s a pleasant day overall, but if you are heading out late in the evening, it might be a good idea to carry an umbrella just in case of a quick pre-monsoon shower.


Without acess to external data or interface to real world, the LLM will generate a hallucinated response.

We can overcome this by augmeting LLMs with access to tools, which can LLM can use to query the data and generate the right response.

## Weather API

An example API to get realtime weather





In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


def get_weather_data(c_lat, c_long):

  url = "https://api.open-meteo.com/v1/forecast"
  params = {
    "latitude": c_lat,
    "longitude": c_long,
    "hourly": "temperature_2m",
    "past_days": 0,
    "forecast_days": 7,
  }
  responses = openmeteo.weather_api(url, params = params)

  response = responses[0]
  print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
  print(f"Elevation: {response.Elevation()} m asl")
  print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")


  hourly = response.Hourly()
  hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

  hourly_data = {"date": pd.date_range(
    start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
    freq = pd.Timedelta(seconds = hourly.Interval()),
    inclusive = "left"
  )}

  hourly_data["temperature_2m"] = hourly_temperature_2m

  hourly_dataframe = pd.DataFrame(data = hourly_data)

  return hourly_dataframe.to_dict(orient = 'records')[:24]


In [ ]:
get_weather_data(12.97, 77.580)

Coordinates: 12.970123291015625°N 77.56363677978516°E
Elevation: 919.0 m asl
Timezone difference to GMT+0: 0s


[{'date': Timestamp('2026-05-04 00:00:00+0000', tz='UTC'),
  'temperature_2m': 21.899999618530273},
 {'date': Timestamp('2026-05-04 01:00:00+0000', tz='UTC'),
  'temperature_2m': 22.299999237060547},
 {'date': Timestamp('2026-05-04 02:00:00+0000', tz='UTC'),
  'temperature_2m': 23.350000381469727},
 {'date': Timestamp('2026-05-04 03:00:00+0000', tz='UTC'),
  'temperature_2m': 25.700000762939453},
 {'date': Timestamp('2026-05-04 04:00:00+0000', tz='UTC'),
  'temperature_2m': 28.450000762939453},
 {'date': Timestamp('2026-05-04 05:00:00+0000', tz='UTC'),
  'temperature_2m': 30.600000381469727},
 {'date': Timestamp('2026-05-04 06:00:00+0000', tz='UTC'),
  'temperature_2m': 31.549999237060547},
 {'date': Timestamp('2026-05-04 07:00:00+0000', tz='UTC'),
  'temperature_2m': 32.099998474121094},
 {'date': Timestamp('2026-05-04 08:00:00+0000', tz='UTC'),
  'temperature_2m': 32.20000076293945},
 {'date': Timestamp('2026-05-04 09:00:00+0000', tz='UTC'),
  'temperature_2m': 33.0},
 {'date': Times

## Integrating Weather tool with LLM

### Run a LLM with tool augmented



In [ ]:
!uv pip install --force-reinstall numpy==1.26.4

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 36ms
Prepared 1 package in 313ms
Uninstalled 1 package in 16ms
Installed 1 package in 32ms
 - numpy==2.4.4
 + numpy==1.26.4


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", device_map="auto")

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

I am Qwen, a large-scale language model independently developed by the Tongyi Lab under Alibaba Group. I am proficient in multiple languages and can assist with various tasks such as answering questions, creating text


### Free-form tool call

In [ ]:
prompt = """
You are an AI assistant that can call tools.

Available tools:

1. get_weather_data(c_lat: float, c_long: float)
   - Get weather data for a given location


First think step by step and write your reasoning inside <thinking>...</thinking>.

When you need to use a tool, output ONLY in the following format and also add your reasoning:

<tool_call>
{
  "name": "<tool_name>",
  "arguments": { ... }
}
</tool_call>

If no tool is needed, respond normally.

---

Example:

User: What's the weather in Delhi?

<tool_call>
{
  "name": "get_weather_data",
  "arguments": {
    "c_lat": 28.6139,
    "c_long": 77.2090
  }
}
</tool_call>

"""

query = "What's the weather like in Bangalore?"

messages = [
    {"role": "system", "content": prompt},
    {"role" : "user", "content" : query}
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=400)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

print(response)


<thinking>
To determine the weather in Bangalore, I need to get the weather data for its geographic coordinates. Bangalore's approximate latitude is 12.9716 and longitude is 77.5946. I will use the get_weather_data tool to fetch the weather information for this location.
</thinking>
<tool_call>
{
  "name": "get_weather_data",
  "arguments": {
    "c_lat": 12.9716,
    "c_long": 77.5946
  }
}
</tool_call><|im_end|>


In [ ]:
import re

import re
import json

def extract_tool_call(text):
    """
    Extracts the first <tool_call>...</tool_call> block and parses JSON.
    Returns dict like {"name": ..., "arguments": {...}} or None.
    """

    pattern = r"<tool_call>\s*(\{.*?\})\s*</tool_call>"
    match = re.search(pattern, text, re.DOTALL)

    if not match:
        return None

    raw_json = match.group(1).strip()

    # Optional: minor cleanup for common LLM issues
    # (trailing commas, smart quotes, etc.)
    raw_json = raw_json.replace("“", '"').replace("”", '"')

    try:
        return json.loads(raw_json)
    except json.JSONDecodeError:
        # simple repair: remove trailing commas
        try:
            cleaned = re.sub(r",\s*}", "}", raw_json)
            cleaned = re.sub(r",\s*]", "]", cleaned)
            return json.loads(cleaned)
        except:
            return None




tool_call = extract_tool_call(response)

print(tool_call)



{'name': 'get_weather_data', 'arguments': {'c_lat': 12.9716, 'c_long': 77.5946}}


In [ ]:
if tool_call:
    result = get_weather_data(**tool_call["arguments"])
    print("Tool result:", result)


Coordinates: 12.970123291015625°N 77.56363677978516°E
Elevation: 910.0 m asl
Timezone difference to GMT+0: 0s
Tool result: [{'date': Timestamp('2026-05-04 00:00:00+0000', tz='UTC'), 'temperature_2m': 21.899999618530273}, {'date': Timestamp('2026-05-04 01:00:00+0000', tz='UTC'), 'temperature_2m': 22.299999237060547}, {'date': Timestamp('2026-05-04 02:00:00+0000', tz='UTC'), 'temperature_2m': 23.350000381469727}, {'date': Timestamp('2026-05-04 03:00:00+0000', tz='UTC'), 'temperature_2m': 25.700000762939453}, {'date': Timestamp('2026-05-04 04:00:00+0000', tz='UTC'), 'temperature_2m': 28.450000762939453}, {'date': Timestamp('2026-05-04 05:00:00+0000', tz='UTC'), 'temperature_2m': 30.600000381469727}, {'date': Timestamp('2026-05-04 06:00:00+0000', tz='UTC'), 'temperature_2m': 31.549999237060547}, {'date': Timestamp('2026-05-04 07:00:00+0000', tz='UTC'), 'temperature_2m': 32.099998474121094}, {'date': Timestamp('2026-05-04 08:00:00+0000', tz='UTC'), 'temperature_2m': 32.20000076293945}, {'da

In [ ]:
# send weather data to the model again for analysis
from pprint import pprint

messages.append({
    "role": "system",
    "content": response
} )
messages.append({"role": "tool", "name": "get_weather_data", "content": json.dumps(result, default=str)})


# Adding context for the final output
messages.append({
    "role": "user",
    "content": "Generate a  sarcastic summary of this weather data."
})

print("\n--- FINAL AGENT MESSAGE ---")
pprint(messages)




--- FINAL AGENT MESSAGE ---
[{'content': '\n'
             'You are an AI assistant that can call tools.\n'
             '\n'
             'Available tools:\n'
             '\n'
             '1. get_weather_data(c_lat: float, c_long: float)\n'
             '   - Get weather data for a given location\n'
             '\n'
             '\n'
             'First think step by step and write your reasoning inside '
             '<thinking>...</thinking>.\n'
             '\n'
             'When you need to use a tool, output ONLY in the following format '
             'and also add your reasoning:\n'
             '\n'
             '<tool_call>\n'
             '{\n'
             '  "name": "<tool_name>",\n'
             '  "arguments": { ... }\n'
             '}\n'
             '</tool_call>\n'
             '\n'
             'If no tool is needed, respond normally.\n'
             '\n'
             '---\n'
             '\n'
             'Example:\n'
             '\n'
             "User: What'

In [ ]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=400)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

print(response)


Oh, absolutely *perfect* weather in Bangalore — a delightful 21.9°C in the morning, then it *soars* to a scorching 33.4°C by noon, only to drop to a chilly 23°C by night. It’s like nature’s way of saying, “Welcome to India — where the temperature is always trying to decide whether it wants to be a tropical paradise or a midday sauna.” Truly, the weather is *flawless* — just what you’d expect from a city that’s always on fire, both literally and in the heat of the moment. 😏🔥<|im_end|>


## Structured Tool calling with a local model

In [ ]:
## TODO

### Tool use with a cloud model

In [ ]:
!uv pip install litellm

Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 1.04s
Prepared 10 packages in 1.02s
Uninstalled 8 packages in 186ms
Installed 10 packages in 121ms
 - aiohttp==3.13.5
 + aiohttp==3.13.4
 - click==8.3.3
 + click==8.1.8
 + fastuuid==0.14.0
 - importlib-metadata==8.7.1
 + importlib-metadata==8.5.0
 - jsonschema==4.26.0
 + jsonschema==4.23.0
 + litellm==1.83.14
 - openai==2.32.0
 + openai==2.24.0
 - pydantic==2.12.3
 + pydantic==2.12.5
 - pydantic-core==2.41.4
 + pydantic-core==2.41.5
 - typer==0.24.2
 + typer==0.23.1


### Running an Agent loop Step-by-Step


In [ ]:
from pprint import pprint
from litellm import completion
from google.colab import userdata
import os

# Set your Gemini API key from Colab's user data
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather_data",
        "description": "Get weather data for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "c_lat": {"type": "number", "description": "Latitude of the location."},
                "c_long": {"type": "number", "description": "Longitude of the location."}
            },
            "required": ["c_lat", "c_long"]
        }
    }
}]


# --- STEP 1: THINKING (Tool Selection) ---
print("--- STEP 1: AGENT IS THINKING ---")
query = "What's the weather like in Delhi?"
messages = [{"role": "user", "content": query}]

response = completion(
    model="gemini/gemini-flash-latest",
    messages=messages,
    tools=tools,
    reasoning_effort="medium"
)

print("\n--- AGENT REASONING ---")
# Accessing the reasoning tokens specifically
print(response.choices[0].message.reasoning_content)

print("\n--- FULL RESPONSE OBJECT (PRETTY PRINTED) ---")
# Use model_dump() instead of deprecated dict()
pprint(response.choices[0].model_dump())

--- STEP 1: AGENT IS THINKING ---

--- AGENT REASONING ---
**Delhi Weather Request: A Logical Breakdown**

Okay, so the user wants the weather in Delhi. Simple enough, but I need to figure out the best way to get that information. My first thought is to use a weather data tool, something like `get_weather_data`. I know that tool requires latitude and longitude as input. So, before I can query the tool, I need to pinpoint Delhi's precise location. A quick search should do the trick. Ah, here we go... Delhi, India: Latitude 28.6139, Longitude 77.2090. Now that I have the necessary coordinates, I can proceed with calling the weather tool and delivering the information.




--- FULL RESPONSE OBJECT (PRETTY PRINTED) ---
{'finish_reason': 'tool_calls',
 'index': 0,
 'message': {'content': None,
             'function_call': None,
             'images': [],
             'provider_specific_fields': {'thought_signatures': ['EvUCCvICAQw51scz6jZiYUG9zCl73Z+Q7eZahnbYJnHCbvn3AwStgR1ZqYmIZaYD1gwmPi4

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]
print(f"Agent decided to call: {tool_call.function.name}")
print(f"Arguments extracted: {tool_call.function.arguments}")

Agent decided to call: get_weather_data
Arguments extracted: {"c_lat": 28.61, "c_long": 77.23}


In [ ]:
# --- STEP 2: ACTING (Executing the Tool) ---
print("\n--- STEP 2: AGENT IS ACTING ---")

# 1. Parse the function name and arguments dynamically
function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)

print(f"Agent is looking for function: {function_name}")

try:
    # Get the function reference from the global namespace
    function_to_call = globals()[function_name]


    raw_weather_data = function_to_call(**function_args)

    print(f"\nSuccessfully executed {function_name}!")
    print(f"Retrieved {len(raw_weather_data)} records.")
except KeyError:
    print(f"Error: The function '{function_name}' does not exist in the notebook.")
except Exception as e:
    print(f"An error occurred during execution: {e}")


--- STEP 2: AGENT IS ACTING ---
Agent is looking for function: get_weather_data
Coordinates: 28.576448440551758°N 77.18678283691406°E
Elevation: 214.0 m asl
Timezone difference to GMT+0: 0s

Successfully executed get_weather_data!
Retrieved 24 records.


In [ ]:
print(raw_weather_data)

[{'date': Timestamp('2026-04-30 00:00:00+0000', tz='UTC'), 'temperature_2m': 26.350000381469727}, {'date': Timestamp('2026-04-30 01:00:00+0000', tz='UTC'), 'temperature_2m': 27.549999237060547}, {'date': Timestamp('2026-04-30 02:00:00+0000', tz='UTC'), 'temperature_2m': 29.200000762939453}, {'date': Timestamp('2026-04-30 03:00:00+0000', tz='UTC'), 'temperature_2m': 31.950000762939453}, {'date': Timestamp('2026-04-30 04:00:00+0000', tz='UTC'), 'temperature_2m': 34.150001525878906}, {'date': Timestamp('2026-04-30 05:00:00+0000', tz='UTC'), 'temperature_2m': 35.5}, {'date': Timestamp('2026-04-30 06:00:00+0000', tz='UTC'), 'temperature_2m': 35.849998474121094}, {'date': Timestamp('2026-04-30 07:00:00+0000', tz='UTC'), 'temperature_2m': 37.70000076293945}, {'date': Timestamp('2026-04-30 08:00:00+0000', tz='UTC'), 'temperature_2m': 37.150001525878906}, {'date': Timestamp('2026-04-30 09:00:00+0000', tz='UTC'), 'temperature_2m': 35.150001525878906}, {'date': Timestamp('2026-04-30 10:00:00+0000

In [ ]:

print("\n--- STEP 3: AGENT IS REPORTING ---")

# We provide the LLM with the results of its own 'Action'
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "name": "get_weather_data",
    "tool_call_id": tool_call.id,
    "content": json.dumps(raw_weather_data, default=str)
})

# Adding context for the final output
messages.append({
    "role": "system",
    "content": "Generate a  sarcastic summary of this weather data."
})

print("\n--- FINAL AGENT MESSAGE ---")
pprint(messages)


--- STEP 3: AGENT IS REPORTING ---

--- FINAL AGENT MESSAGE ---
[{'content': "What's the weather like in Delhi?", 'role': 'user'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'EvICCu8CAQw51sfo18S0vH2GkjPobj4l+by7Dn4AlW9qRSmIN3NSugbzQ4fDBNW8CZc/h12pH2cdzJwFxpBmms4WBGsarrPCq/SO7LJ0tl+YgLB6Ujs676F+0BP9YGqohqDLQiOBuhJzTfCc91t184+d+jKt22i/nJ1+GKXRVun5RtFi8onCU5NPok6NFdAgCVNVAX/zTVIJ6DaeTNw+jNzG6Jsk9v+lQaHEMdCfWMFVbOWrmMthWJNJ6R1BKr7cFs1pixJXZif7/ZGNrdgol0JOjKkdqMF39vYZDS+Vxr3zMnyiW+g7eSALnCG+nPUYwQPL04J+EGLT5D7S6hTh8brEckt0DCFvqi6b/u5w0eOEG0DBgny45ClJzqw9ISFCxSomeapOn5TDZXaUmUwbei8rjnnpLHx1kCy1VN/91RBGryvjesi1b3dgvZfdl0f/NKWPUU1Ct1xzKsboqCA5GTVwZDO0Z5rn4absVjCOAKIUrh79PQ=='}, function=Function(arguments='{"c_lat": 28.6139, "c_long": 77.209}', name='get_weather_data'), id='call_70d52c2982ea4ab3be1908713dc5__thought__EvICCu8CAQw51sfo18S0vH2GkjPobj4l+by7Dn4AlW9qRSmIN3NSugbzQ4fDBNW8CZc/h12pH2cdzJwFxp

In [ ]:

final_report = completion(
    model="gemini/gemini-flash-latest",
    messages=messages
)

print("Final  Output:")
print(final_report)

Final  Output:
ModelResponse(id='XW3zadPfM9qc_uMP59TfgQg', created=1777560921, model='gemini-flash-latest', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Oh, it’s just a delightful spring day in Delhi! Because nothing says "refreshing" like a crisp 26°C... at midnight. \n\nBy mid-morning, the temperature generously climbs to a brisk 37.7°C (about 100°F), which is perfect for anyone whose dream is to live inside a convection oven. If you’ve ever wondered what it’s like to be a rotisserie chicken, today is your day! \n\nDon\'t worry, though—the evening offers a "chilling" reprieve, dropping all the way down to a cool 25°C. It’s basically a winter wonderland, provided your idea of a snowflake is a soot particle and your idea of a breeze is the exhaust from a stationary bus. Pack a sweater, or perhaps just some industrial-strength sunscreen and a portable walk-in freezer. Enjoy!', role='assistant', tool_calls=Non

## Coding Agent Implmentation

In [ ]:
import os
import json
from litellm import completion
from google.colab import userdata

# --- TOOL DEFINITION ---

def view(path: str) -> str:
    """Reads a file or lists a directory to understand the codebase."""
    if not os.path.exists(path):
        return f"Error: Path '{path}' does not exist."

    if os.path.isdir(path):
        try:
            files = os.listdir(path)
            return f"Directory listing for {path}:\n" + "\n".join(files)
        except Exception as e:
            return f"Error listing directory: {e}"
    else:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            return f"Error reading file: {e}"

# --- AGENT LOGIC ---

def run_explorer_agent(query: str, repo_root: str = "."):
    tools = [{
        "type": "function",
        "function": {
            "name": "view",
            "description": "Read a file or list directory contents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path to file or directory"}
                },
                "required": ["path"]
            }
        }
    }]

    messages = [
        {"role": "system", "content": f"You are a read-only code explorer. Your goal is to answer questions about the repository at {repo_root}. Use the 'view' tool to browse files. You have a strict limit of 10 steps and 20 file reads."},
        {"role": "user", "content": query}
    ]

    step_limit = 10
    read_count = 0
    max_reads = 20
    token_limit = 8000

    for step in range(step_limit):
        print(f"\n--- STEP {step + 1} ---")

        response = completion(
            model="gemini/gemini-flash-latest",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            reasoning_effort="medium"
        )

        res_msg = response.choices[0].message
        messages.append(res_msg)

        # Print Reasoning / Thought if available
        if hasattr(res_msg, 'reasoning_content') and res_msg.reasoning_content:
            print(f"Agent Reasoning: {res_msg.reasoning_content}")

        if res_msg.content:
            print(f"Agent Response: {res_msg.content}")

        if not res_msg.tool_calls:
            print("\n--- FINAL ANSWER ---")
            return res_msg.content

        for tool_call in res_msg.tool_calls:
            if read_count >= max_reads:
                messages.append({"role": "tool", "tool_call_id": tool_call.id, "name": "view", "content": "Error: Max read limit reached."})
                continue

            args = json.loads(tool_call.function.arguments)
            path = args.get('path')

            print(f"Action: view('{path}')")
            result = view(path)
            read_count += 1

            if len(result) > token_limit:
                result = result[:token_limit] + "\n... [Truncated due to context limits]"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "view",
                "content": result
            })

    return "Agent reached maximum step limit without a final answer."


In [ ]:
!git clone https://github.com/SWE-agent/mini-swe-agent.git

Cloning into 'mini-swe-agent'...
remote: Enumerating objects: 16952, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 16952 (delta 2), reused 2 (delta 2), pack-reused 16946 (from 2)
Receiving objects: 100% (16952/16952), 17.41 MiB | 17.94 MiB/s, done.
Resolving deltas: 100% (9933/9933), done.


In [ ]:
query = "What files are in the mini-swe-agent directory and what is thios repo doing?"
print(run_explorer_agent(query))


--- STEP 1 ---
Action: view('.')

--- STEP 2 ---


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...Jb', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Action: view('mini-swe-agent')

--- STEP 3 ---


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...bp', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Action: view('mini-swe-agent/README.md')

--- STEP 4 ---


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...8x', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Agent Response: The `mini-swe-agent` directory contains a minimal, yet powerful, AI software engineering agent designed to solve real-world coding tasks (like those in SWE-bench). 

### Key Files and Directories
*   **`src/`**: Contains the core logic:
    *   `src/minisweagent/agents/`: Implementation of the agent class (designed to be around 100 lines of Python).
    *   `src/minisweagent/environments/`: Adapters for running code in different environments (Local, Docker, Podman, Singularity, etc.).
    *   `src/minisweagent/models/`: Model wrappers (e.g., via `litellm`) to support various LLMs.
*   **`tests/`**: Unit and integration tests.
*   **`docs/`**: Documentation files.
*   **`README.md`**: Overview, features, and quick-start guide.
*   **`pyproject.toml`**: Dependency and package configuration.
*   **`mkdocs.yml`**: Configuration for the documentation site.
*   **`.pre-commit-config.yaml`**: Linting and formatting rules.

### What is this repo doing?
The repository provides a

In [ ]:
query = "Can you plsn for removal of litellm?"
print(run_explorer_agent(query))


--- STEP 1 ---
Agent Reasoning: **My Litellm Removal Plan**

Okay, so the goal is clear: get rid of `litellm` from this repo. This is going to require a methodical approach, no surprises there. My first instinct, and the right one, is to systematically understand the scope of the problem.

1.  **Locate all instances:** I need to pinpoint everywhere `litellm` is actually utilized in the codebase. A simple search for "litellm" will be the starting point, of course, but I'll need to be thorough. I'll need to make sure to search case-insensitively, in comments, and potentially within generated files if those are present. The goal is to leave no stone unturned.

2.  **Dependency deep-dive:** Once I know *where* it's used, I need to know *what* it's bringing in with it. What are its related dependencies? Are they core to some other functionality, or are they isolated? I'll need to map out the dependency graph for `litellm` to understand the ripple effect of its removal. I'll likely use some

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...o=', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Agent Reasoning: Here's my take on the code exploration, tailored for an expert's mind:

**Initial Assessment: Deep Dive into mini-swe-agent**

Alright, the initial code landscape is becoming clearer. The architecture seems to be centralized around `mini-swe-agent`, which is a pretty common pattern. Given the name, I'm expecting this to be the core component for the "SWE" – likely the software engineering aspects, the agent's brain, so to speak. Let's start by dissecting the internals of that directory. I'll be looking for things like: the main entry points, the agent's data structures, interaction with external tools or APIs, and, critically, the control flow and decision-making logic. My initial hypothesis is that it likely implements a state machine, or at least a highly modular workflow. I need to understand how this agent interacts with, and interprets results from the environment. Knowing how it handles edge cases, error conditions, and the ability to self correct. From an optimi

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...uC', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Agent Reasoning: Okay, here's my summary, taking the perspective of someone deeply familiar with software project management and dependency analysis:

**Tracking Down a Dependency**

Alright, so `litellm` is showing up as a dependency listed in `project.dependencies`. This means it's a critical component of the project, something we *rely* on. Now, my immediate concern is always understanding the scope of that dependency.

I need to quickly figure out *where* this library is being used within the codebase. What files, what modules, what specific functions are invoking or importing `litellm`? Is it used directly in core logic, or tucked away in some utility scripts? And if it's imported, is it done through a direct `import litellm`, or is it buried within another import?

My goal is to efficiently find all instances to get a sense of how deeply intertwined `litellm` is, so I can accurately assess its impact. This is crucial for things like security vulnerability scanning, upgrade impact

...

Agent Reasoning: Here's my thought process summarized:

**Navigating the Codebase**

Okay, let's get down to it. I'm focusing on finding the `litellm` integration within the `mini-swe-agent` project. Specifically, I'm digging into the `src/minisweagent` directory. This suggests I'm looking for the core components related to the agent's interaction with `litellm`. The file structure implies that `minisweagent` houses the main logic or the primary integration point. My immediate goal is to locate where this library is called, how it's configured, and what functions rely on it. I need to understand the connection between the agent and `litellm` so I can better understand its architecture.



Action: view('mini-swe-agent/src/minisweagent')

--- STEP 6 ---
Agent Reasoning: Okay, here's my summary, taking on the persona you've described:

**Prioritizing the LLM Code Search**

Alright, let's get down to brass tacks. Where's the logical starting point for investigating the LLM interaction with

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...en', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Agent Reasoning: Here's my thought process, as it stands:

**Initial Assessment of the `litellm` Codebase**

Alright, I'm diving into this `litellm` project. First things first, let's get a lay of the land. I've spotted a few key files jumping out at me: `litellm_model.py`, `litellm_textbased_model.py`, and `litellm_response_model.py`. Those file names alone give me some initial clues – `model` suggests core architecture or interface, `textbased` likely handles the text processing component, and `response_model` points towards managing the structure of outputs.

Before I go too deep into any specific file, I need the bigger picture. I'll need to figure out the scope of this `litellm` entity. I want to see how these pieces fit together. My next step will be to explore other files and determine where this `litellm` code is being utilized. Understanding its integration within the broader system is going to be crucial for any meaningful analysis or modification.



Action: view('mini-swe-a

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...I=', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Agent Reasoning: **Examining the Model Class Defaults and Mappings**

Okay, so the `get_model_class` function seems to default to `LitellmModel`. That's interesting, but I need to understand the implications of this default. Right now, it's just importing `LitellmModel` directly.

My immediate thought is, what are the other models defined in `_MODEL_CLASS_MAPPING` doing? Are `OpenRouterModel` and the other entries also relying on `litellm` under the hood? It's highly probable, given the name, but I need to confirm this. Understanding their implementation is crucial to assess the extent of `litellm`'s influence and ensure compatibility or make necessary adjustments if it is removed or modified.



Action: view('mini-swe-agent/src/minisweagent/models/openrouter_model.py')

--- STEP 9 ---
Agent Reasoning: Here's my summary, as I see it:

**Digging into Model Dependencies**

Okay, so I'm taking a look at these models, trying to understand their architecture and how they interact with other